![Image](./snapshot_strategy.png)

1차에서 수행한 100개의 질문을 제외하고 다시 랜덤하게 100개 추출


In [ ]:
# pip install openpyxl 

In [1]:
import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

In [5]:
import psycopg2
import pandas as pd
import numpy as np
import config.config as conf
import pickle
import lib.preprocess.preprocess as pp
import lib.preprocess.SectionExtractor as se
import lib.annotation.D_Annotation as da
import lib.annotation.Self_Consistency as sc
import re
import datetime
import pandas as pd
import re
import numpy as np
from sklearn import metrics


In [4]:
htmlp = pp.HTMLParser()
codep = pp.CodeSectionParser()
ts = se.SectionExtractor()

In [52]:
file_list = os.listdir('.')

In [55]:
file_list = [x for x in file_list if x.endswith('snapshop2_sample.csv')]

In [76]:
df_q = pd.DataFrame()
for file in file_list :
    df_q = pd.concat([df_q, pd.read_csv(file, index_col=False)], axis = 0)

In [78]:
df_q[['id', 'question']]

,id,question
0,70264389,<Title>Population pyramid with seaborn python<...
1,70274885,"<Title>insert or update on table ""django_admin..."
2,70313318,<Title>perform upsert operation on postgres li...
3,70540832,<Title>Python/SQL - Connecting to different da...
4,70693775,<Title>How to pass an object to a process crea...
...,...,...
103,70731352,<Title>Python str not list</Title>. <Question>...
104,74047007,<Title>How to detect black contour in image us...
105,78840333,<Title>How to get every combination possible i...
106,71281263,<Title>Loop through a directory and add filena...


In [79]:
path_list = [f'../golden_dataset/{x}' for x in ['2nd', '3rd', '5th']]

In [80]:
a_list = []
for path in path_list : 
    a_list.append([f'{path}/{x}' for x in os.listdir(path) if x.endswith('.xlsx') and not x.startswith('~')])


In [81]:
mapping = {'Basic': '<Difficulty Level>0</Difficulty Level>', 
           'Intermediate': '<Difficulty Level>1</Difficulty Level>', 
           'Advanced' : '<Difficulty Level>2</Difficulty Level>'}

In [82]:
tot_df = pd.DataFrame()
for a in a_list:
    df_0 = pd.read_excel(f'{a[0]}', engine='openpyxl')
    df_1 = pd.read_excel(f'{a[1]}', engine='openpyxl')
    df_2 = pd.read_excel(f'{a[2]}', engine='openpyxl')
    df_3 = pd.read_excel(f'{a[3]}', engine='openpyxl')

    df_0 = df_0[['id', 'answer']].rename(columns={'answer' : 'a_jh'})
    df_1 = df_1[['id', 'answer']].rename(columns={'answer' : 'a_hj'})
    df_2 = df_2[['id', 'answer']].rename(columns={'answer' : 'a_jw'})
    df_3 = df_3[['id', 'answer']].rename(columns={'answer' : 'a_mk'})

    df_0['a_jh'] = df_0['a_jh'].map(mapping)
    df_1['a_hj'] = df_1['a_hj'].map(mapping)
    df_2['a_jw'] = df_2['a_jw'].map(mapping)
    df_3['a_mk'] = df_3['a_mk'].map(mapping)

    df_m = df_0.merge(df_1, on='id') \
                .merge(df_2, on='id') \
                .merge(df_3, on='id')
    
    df_m['sum'] = (df_m['a_jh']==df_m['a_hj'])&(df_m['a_hj'] ==df_m['a_jw']) & (df_m['a_jw']==df_m['a_mk'])
    df_c = df_m[df_m['sum'] ==True]
    tot_df = pd.concat([tot_df, df_c], axis = 0)

In [83]:
df = tot_df[['id', 'a_jh']].rename(columns={'a_jh' : 'answer'})

In [86]:
df_golden = pd.merge(df_q[['id', 'question']], df, on = 'id')

In [88]:
df_golden.to_csv('/usr/share/d_ollama/data/q_output_code_y_snapshot2.csv', index=False)

In [4]:
pd.read_csv('/usr/share/d_ollama/data/q_output_code_y_snapshot2.csv')['id']

0      71389500
1      72118859
2      72422859
3      70266683
4      72329302
         ...   
99     78310990
100    71698879
101    70731352
102    74047007
103    71281263
Name: id, Length: 104, dtype: int64